# Evaluation and error analysis (Section 5) - working notebook

This notebook holds the cells for **Section 5: Evaluate and explain** of the project guide.
They will be merged into `project.ipynb`.

| Part | Merge into project.ipynb? | Purpose |
|---|---|---|
| **A. Development setup** | **No** | Stand-ins for Sections 1-4 (data, baseline, the Section 4 selection variables) so this notebook runs on its own |
| **B. Section 5** | **Yes**, directly after Section 4 | Loss curves, the single test evaluation, confusion matrix, example images, discussion |
| **C. Validation dry run** | **No** | Runs the same reporting code on *validation* data to check the output without touching the test set |

The test-set cell runs only when Section 4 reports a **final** selection
(`SELECTION_IS_FINAL = True`). With the stand-ins in Part A the selection is provisional,
so the test cells print "skipped" and the test images are never loaded.

**Variables Section 5 expects from earlier sections**

| Variable | From | Meaning |
|---|---|---|
| `DATA_DIR`, `load_image_split`, `y_val` | Data preparation | Shared data loading |
| `predict_labels`, `defect_metrics`, `DECISION_THRESHOLD` | Section 4 helpers | Fixed 0.5 rule and metrics |
| `selected_candidate`, `selected_name`, `selected_predict_proba` | Section 4 | The frozen selected model |
| `SELECTION_IS_FINAL`, `assert_selection_unchanged()` | Section 4 | Guard against early or repeated selection |
| `cnn_base_candidate`, `cnn_variant_candidate` | CNN sections | Registered CNN runs |
| `cnn_base_history`, `cnn_variant_history` | CNN sections | `{"train_loss": [...], "val_loss": [...]}`, one value per epoch |

## Part A - Development setup (do not merge)

Re-creates the data split and the logistic regression baseline with the same settings as
`project.ipynb`, plus minimal copies of the Section 4 helpers. The selection is marked
**provisional** because the CNN runs are not available yet.

In [ ]:
# DEV ONLY - project.ipynb defines all of this in Sections 1-4.
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, precision_score, recall_score,
)

SEED = 770
IMAGE_SIZE = (64, 64)


def find_data_dir():
    configured = os.environ.get("CASTING_DATA_DIR")
    options = (
        [Path(configured)] if configured
        else [Path("data") / "casting_512x512", Path("data")]
    )
    for option in options:
        if (option / "def_front").is_dir() and (option / "ok_front").is_dir():
            return option
    raise FileNotFoundError(
        "Could not find def_front/ and ok_front/ in: "
        + ", ".join(str(option) for option in options)
    )


DATA_DIR = find_data_dir()
manifest = pd.read_csv(Path("splits") / "split_manifest.csv")


def load_image_split(split_name):
    rows = (
        manifest[manifest["split"] == split_name]
        .sort_values("relative_path")
        .reset_index(drop=True)
    )
    images = []
    for relative_path in rows["relative_path"]:
        with Image.open(DATA_DIR / relative_path) as image:
            image = image.convert("L").resize(
                IMAGE_SIZE, resample=Image.Resampling.BILINEAR
            )
            images.append(np.asarray(image, dtype=np.float32) / 255.0)
    return np.stack(images), rows["label"].to_numpy(dtype=np.int64), rows


X_train_images, y_train, train_rows = load_image_split("train")
X_val_images, y_val, val_rows = load_image_split("validation")

baseline_model = LogisticRegression(
    solver="lbfgs", C=1.0, max_iter=2000, tol=1e-4, random_state=SEED,
)
start_time = time.perf_counter()
baseline_model.fit(X_train_images.reshape(len(X_train_images), -1), y_train)
baseline_training_seconds = time.perf_counter() - start_time

# --- Minimal copies of the Section 4 helpers (fairness.ipynb has the full versions) ---
DECISION_THRESHOLD = 0.5
DEFECTIVE, ACCEPTABLE = 1, 0


def predict_labels(defect_probabilities):
    return (np.asarray(defect_probabilities) >= DECISION_THRESHOLD).astype(np.int64)


def defect_metrics(y_true, defect_probabilities):
    y_pred = predict_labels(defect_probabilities)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[ACCEPTABLE, DEFECTIVE]).ravel()
    return {
        "Precision": precision_score(y_true, y_pred, pos_label=DEFECTIVE, zero_division=0),
        "Recall": recall_score(y_true, y_pred, pos_label=DEFECTIVE, zero_division=0),
        "F1": f1_score(y_true, y_pred, pos_label=DEFECTIVE, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "False negatives": int(fn),
        "False positives": int(fp),
    }


def _baseline_predict_proba(images):
    flat = np.asarray(images).reshape(len(images), -1)
    return baseline_model.predict_proba(flat)[:, list(baseline_model.classes_).index(DEFECTIVE)]


# --- Stand-in for the Section 4 result: provisional, baseline only ---
selected_candidate = {
    "name": "Logistic regression",
    "val_probabilities": _baseline_predict_proba(X_val_images),
    "training_seconds": baseline_training_seconds,
    "selected_epoch": None,
    "predict_proba": _baseline_predict_proba,
}
selected_name = selected_candidate["name"]
selected_predict_proba = selected_candidate["predict_proba"]
missing_candidates = ["cnn_base_candidate", "cnn_variant_candidate"]
SELECTION_IS_FINAL = False


def assert_selection_unchanged():
    if not SELECTION_IS_FINAL:
        raise RuntimeError("Selection is provisional. Do not evaluate the test set yet.")


print("Data folder:", DATA_DIR.as_posix())
print("Stand-in selection:", selected_name, "| final:", SELECTION_IS_FINAL)

## 5. Evaluation and error analysis

The validation comparison table for all three configurations (precision, recall, F1 for
the defective class, accuracy and training time) is produced in Section 4, where it is used
for selection. This section adds the CNN loss curves, then evaluates the selected model on
the test set **once**.

Every result below uses the frozen model from Section 4 and the fixed rule
P(defective) ≥ 0.5. Nothing is retrained, retuned or reselected after test results are seen,
and no model is trained on validation data.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

CLASS_NAMES = {ACCEPTABLE: "Acceptable", DEFECTIVE: "Defective"}
MAX_ERROR_EXAMPLES = 6
NUM_CORRECT_EXAMPLES = 2


def metrics_row(name, split_name, y_true, defect_probabilities):
    return {
        "Configuration": name,
        "Split": split_name,
        "Images": len(y_true),
        **defect_metrics(y_true, defect_probabilities),
    }


def plot_confusion_matrix(y_true, y_pred, title):
    fig, ax = plt.subplots(figsize=(4.8, 4.2))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        labels=[ACCEPTABLE, DEFECTIVE],
        display_labels=[f"{CLASS_NAMES[c]} ({c})" for c in (ACCEPTABLE, DEFECTIVE)],
        cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[ACCEPTABLE, DEFECTIVE]).ravel()
    print(f"True negatives  (acceptable kept):     {tn}")
    print(f"False positives (acceptable rejected): {fp}")
    print(f"False negatives (defects accepted):    {fn}")
    print(f"True positives  (defects caught):      {tp}")


def choose_error_examples(y_true, y_pred, probabilities, max_examples=MAX_ERROR_EXAMPLES):
    """Up to max_examples errors, covering both types when available.

    Each type gets up to half the slots; unused slots go to the other type. Within a
    type, the most confident mistakes come first (false negatives with the lowest
    P(defective), false positives with the highest). Ties keep filename order.
    """
    index = np.arange(len(y_true))
    false_negatives = index[(y_true == DEFECTIVE) & (y_pred == ACCEPTABLE)]
    false_positives = index[(y_true == ACCEPTABLE) & (y_pred == DEFECTIVE)]
    false_negatives = false_negatives[np.argsort(probabilities[false_negatives], kind="stable")]
    false_positives = false_positives[np.argsort(-probabilities[false_positives], kind="stable")]

    half = max_examples // 2
    n_fn = min(len(false_negatives), max(half, max_examples - len(false_positives)))
    n_fp = min(len(false_positives), max_examples - n_fn)
    return np.concatenate([false_negatives[:n_fn], false_positives[:n_fp]]).astype(int)


def choose_correct_examples(y_true, y_pred, probabilities, count=NUM_CORRECT_EXAMPLES):
    """One confidently correct image per class when possible (defect caught, good part kept)."""
    index = np.arange(len(y_true))
    true_positives = index[(y_true == DEFECTIVE) & (y_pred == DEFECTIVE)]
    true_negatives = index[(y_true == ACCEPTABLE) & (y_pred == ACCEPTABLE)]
    true_positives = true_positives[np.argsort(-probabilities[true_positives], kind="stable")]
    true_negatives = true_negatives[np.argsort(probabilities[true_negatives], kind="stable")]

    chosen = list(true_positives[:1]) + list(true_negatives[:1])
    leftovers = list(true_positives[1:]) + list(true_negatives[1:])
    chosen += leftovers[: max(0, count - len(chosen))]
    return np.array(chosen[:count], dtype=int)


def outcome_name(true_label, predicted_label):
    if true_label == predicted_label:
        return "Correct"
    if true_label == DEFECTIVE:
        return "False negative (defect accepted)"
    return "False positive (acceptable rejected)"


def show_examples(rows, y_true, y_pred, probabilities, indices, title, columns=3):
    """Show the original 512x512 photographs (the model saw 64x64 grayscale versions)."""
    if len(indices) == 0:
        print(f"{title}: none to show.")
        return
    n_rows = int(np.ceil(len(indices) / columns))
    fig, axes = plt.subplots(n_rows, columns, figsize=(4.2 * columns, 4.8 * n_rows),
                             squeeze=False, layout="constrained")
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), indices):
        relative_path = rows.loc[i, "relative_path"]
        with Image.open(DATA_DIR / relative_path) as image:
            ax.imshow(image.convert("L"), cmap="gray")
        ax.set_title(
            f"{outcome_name(y_true[i], y_pred[i])}\n"
            f"True: {CLASS_NAMES[y_true[i]]} | Predicted: {CLASS_NAMES[y_pred[i]]}\n"
            f"P(defective) = {probabilities[i]:.3f}\n{Path(relative_path).name}",
            fontsize=9,
        )
    fig.suptitle(title)
    plt.show()

### 5.1 Training and validation loss by epoch (both CNN runs)

Both runs are drawn on the same y-axis so they can be compared directly. The dashed line
marks the checkpoint chosen in Section 4 from validation F1.

In [ ]:
loss_runs = []
for candidate_variable, history_variable in [
    ("cnn_base_candidate", "cnn_base_history"),
    ("cnn_variant_candidate", "cnn_variant_history"),
]:
    if candidate_variable in globals() and history_variable in globals():
        loss_runs.append((globals()[candidate_variable], globals()[history_variable]))
    else:
        print(f"Not available yet: {candidate_variable} / {history_variable}")

if loss_runs:
    fig, axes = plt.subplots(1, len(loss_runs), figsize=(6 * len(loss_runs), 4.2),
                             sharey=True, squeeze=False, layout="constrained")
    for ax, (candidate, history) in zip(axes[0], loss_runs):
        epochs = np.arange(1, len(history["train_loss"]) + 1)
        ax.plot(epochs, history["train_loss"], marker="o", label="Training loss")
        ax.plot(epochs, history["val_loss"], marker="o", label="Validation loss")
        if candidate.get("selected_epoch") is not None:
            ax.axvline(candidate["selected_epoch"], color="gray", linestyle="--",
                       label=f"Selected checkpoint (epoch {candidate['selected_epoch']})")
        ax.set_title(candidate["name"])
        ax.set_xlabel("Epoch")
        ax.set_xticks(epochs)
        ax.grid(alpha=0.3)
        ax.legend()
    axes[0, 0].set_ylabel("Loss")
    fig.suptitle("CNN training and validation loss by epoch")
    plt.show()

### 5.2 Test-set evaluation of the selected model (run once)

The test images are loaded here for the first time. The selected, frozen model scores
them once with P(defective) ≥ 0.5. The selected model's validation row is shown beside
its test row for reference only; it is not used to change anything.

In [ ]:
test_results = None

if not SELECTION_IS_FINAL:
    print("SKIPPED: the Section 4 selection is provisional (missing: "
          + ", ".join(missing_candidates) + "). The test set has not been loaded.")
else:
    assert_selection_unchanged()  # same frozen model that was selected on validation

    X_test_images, y_test, test_rows = load_image_split("test")
    test_probabilities = np.asarray(selected_predict_proba(X_test_images), dtype=np.float64).ravel()
    test_predictions = predict_labels(test_probabilities)
    assert test_probabilities.shape == y_test.shape

    test_results = {
        "rows": test_rows,
        "y_true": y_test,
        "probabilities": test_probabilities,
        "predictions": test_predictions,
    }

    test_summary = pd.DataFrame([
        metrics_row(selected_name, "Validation (selection)", y_val,
                    selected_candidate["val_probabilities"]),
        metrics_row(selected_name, "Test (final)", y_test, test_probabilities),
    ]).set_index(["Configuration", "Split"])

    display(test_summary.style.format({
        "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}", "Accuracy": "{:.4f}",
    }).set_caption(
        f"Selected model: {selected_name}. Decision rule: P(defective) ≥ {DECISION_THRESHOLD}. "
        "Precision, recall and F1 are for the defective class."
    ))

In [ ]:
if test_results is None:
    print("SKIPPED: no test results yet.")
else:
    plot_confusion_matrix(
        test_results["y_true"], test_results["predictions"],
        title=f"Test set confusion matrix - {selected_name}",
    )

### 5.3 Misclassified and correctly classified test images

Up to six misclassified test images are shown, covering both error types when both occur.
Each type gets up to three slots, and unused slots go to the other type. Within a type the
most confident mistakes come first. Two correctly classified images follow: one defect
that was caught and one acceptable part that was kept. Titles give the true label, the
predicted label and the model's P(defective). The photographs are the 512 × 512 originals;
the model saw them as 64 × 64 grayscale.

In [ ]:
if test_results is None:
    print("SKIPPED: no test results yet.")
else:
    r = test_results
    error_indices = choose_error_examples(r["y_true"], r["predictions"], r["probabilities"])
    total_errors = int((r["y_true"] != r["predictions"]).sum())

    if total_errors == 0:
        print("The selected model made no errors on the test set, so there are no "
              "misclassified images to show.")
    else:
        print(f"Showing {len(error_indices)} of {total_errors} misclassified test images.")
        show_examples(r["rows"], r["y_true"], r["predictions"], r["probabilities"],
                      error_indices, title="Misclassified test images")

    correct_indices = choose_correct_examples(r["y_true"], r["predictions"], r["probabilities"])
    show_examples(r["rows"], r["y_true"], r["predictions"], r["probabilities"],
                  correct_indices, title="Correctly classified test images", columns=2)

### 5.4 Discussion

*Draft. Replace the bracketed parts with the final numbers and observations once the test
cell has run. Do not change any model or setting after reading the test results.*

**False negatives vs. false positives.** A *false negative* is a defective casting
predicted as acceptable: the defect is missed and the part moves on to assembly or a
customer. A *false positive* is an acceptable casting predicted as defective: a good part
is rejected, and is then scrapped or sent for manual re-inspection. On the test set the
selected model made [__] false negatives and [__] false positives.

For an inspection workflow, **false negatives matter more**. A missed defect can cause a
failure in use, safety risk, returns or a recall, and it is found far from where it was
made. A false positive costs a re-check or a scrapped good part, which is limited and
visible at the inspection station. Defect recall is therefore the most important single
number here. However, a model that rejects too many good parts would overwhelm manual
re-inspection, so precision still matters. The project fixes the threshold at 0.5. In
practice a lower threshold would trade more false positives for fewer false negatives, but
that choice would have to be made on validation data, not test data.

**What the errors show.** [Describe patterns in the misclassified images. For example: are
the missed defects small, low-contrast or near the rim? Are rejected good parts affected by
lighting, glare or surface texture? Could 64 × 64 downsampling have removed the detail?]

**Limitations.**

1. **Controlled photographs from one source.** All images are top-down photos of one type
   of submersible pump impeller from a single publisher, with similar lighting and framing.
   These results do not show the model would work in a new factory, on a different
   product, or with a different camera, lighting or part position. That would need new
   data from those conditions.
2. **Small test set.** The test set has 195 images (117 defective, 78 acceptable). One
   more or one fewer error changes defect F1 by roughly 0.005, so small differences between
   models are within noise. Only one split and one random seed were used.
3. **Class balance differs from production.** Defects make up 60% of this dataset. On a
   real line defects are usually rare, so the same model would have far lower precision
   (more of its rejections would be good parts) than measured here.
4. **Low resolution and exact-only duplicate checks.** Images were reduced to 64 × 64
   grayscale, which may erase small defects. The audit removed only exact duplicates; near
   duplicates (the same part photographed twice) could appear in more than one split and
   inflate scores.

## Part C - Validation dry run (do not merge)

Runs the same reporting functions on the **validation** set with the stand-in model, to
check the table, confusion matrix and example images. Validation results were already
used for selection, so looking at them here does not leak test information. The test set
is not touched.

In [ ]:
# DEV ONLY - validation data, not test data.
dry_probabilities = np.asarray(selected_predict_proba(X_val_images), dtype=np.float64)
dry_predictions = predict_labels(dry_probabilities)

display(pd.DataFrame([
    metrics_row(selected_name, "Validation (dry run)", y_val, dry_probabilities)
]).set_index(["Configuration", "Split"]).round(4))

plot_confusion_matrix(y_val, dry_predictions,
                      title=f"VALIDATION dry run - {selected_name}")

show_examples(val_rows, y_val, dry_predictions, dry_probabilities,
              choose_error_examples(y_val, dry_predictions, dry_probabilities),
              title="VALIDATION dry run - misclassified")
show_examples(val_rows, y_val, dry_predictions, dry_probabilities,
              choose_correct_examples(y_val, dry_predictions, dry_probabilities),
              title="VALIDATION dry run - correct", columns=2)